# Test Summary By Step

This notebook is a map of the main tests already present in the repo. It is designed to reduce config confusion:

- use precomputed results where they already exist
- state clearly which config generated each result family
- compute only lightweight checks on top of cached data
- keep daily exposure explicitly non-lookahead

In [27]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dynamic_factor_copula import (
    compare_apply_returns,
    compare_sp_exposure,
    compare_trend_exposure,
    compute_port_opt_style_metrics,
    curve_from_returns,
    default_paths,
    load_overlay_compare_prices,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
paths = default_paths(ROOT)

def read_csv_clean(path, index_col=None):
    df = pd.read_csv(path, index_col=index_col)
    if "Unnamed: 0" in df.columns:
        df = df.rename(columns={"Unnamed: 0": "Strategy"})
    return df

def yes_no(value):
    return "Yes" if value else "No"

def display_path(path):
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

## Precompute Status

This first section answers two questions:

1. Which result families are already saved as precomputed files?
2. Which families are still computed on the fly inside this summary notebook?

In [28]:
tracked_files = [
    ("Main US sleeve metrics", paths.result_dir / "multi_factor_copula_metrics.csv", "precomputed"),
    ("US lookback sweep", paths.result_dir / "static_hmm_603010_lookback_sweep.csv", "precomputed"),
    ("US rebalance sweep", paths.result_dir / "static_hmm_603010_rebalance_sweep.csv", "precomputed"),
    ("US mix sweep", paths.result_dir / "static_hmm_momentum_mix_sweep.csv", "precomputed"),
    ("US overlay summary", paths.result_dir / "overlay_comparison_summary.csv", "precomputed"),
    ("Confirmed best US overlay", paths.result_dir / "joint_confirm_603010_504d_1m_overlay_summary_usd.csv", "precomputed"),
    ("Thailand PIT sleeve", paths.result_dir / "thai_set100_pit_metrics.csv", "precomputed"),
    ("US/TH stocks-only blend", paths.result_dir / "us_th_stocks_only_blended_summary_thb.csv", "precomputed"),
    ("US/TH stocks-only joint model", paths.result_dir / "us_th_joint_stocks_only_pit_reselect_summary_thb.csv", "precomputed"),
    ("US/TH Gold/BTC blend", paths.result_dir / "us_th_gold_btc_blended_summary_thb.csv", "precomputed"),
    ("US/TH Gold/BTC joint model", paths.result_dir / "us_th_joint_model_summary_thb.csv", "precomputed"),
    ("US/TH objective sweep", paths.result_dir / "us_th_joint_model_objective_sweep_thb.csv", "precomputed"),
    ("US/TH asset-count sweep", paths.result_dir / "us_th_best_cap_asset_count_pit_reselect_sweep_summary_thb.csv", "precomputed"),
    ("US/TH all-asset static caps", paths.result_dir / "us_th_all_asset_static_caps_pit_reselect_summary_thb.csv", "precomputed"),
    ("US/TH all-asset cap sweep", paths.result_dir / "us_th_all_asset_cap_sweep_pit_reselect_summary_thb.csv", "precomputed"),
    ("US/TH top-5 cap objective sweep", paths.result_dir / "us_th_all_asset_cap_top5_objective_pit_reselect_sweep_thb.csv", "precomputed"),
    ("US/TH stocks-only vs Gold/BTC comparison", paths.result_dir / "us_th_stocks_only_vs_gold_btc_side_trigger_pit_reselect_comparison_thb.csv", "precomputed"),
    ("Config registry", paths.result_dir / "test_config_registry.csv", "precomputed"),
    ("SPY buy and hold metrics", paths.source_cache_root / "benchmark.parquet", "computed in notebook from cache"),
    ("Per-asset daily exposure test", paths.local_cache_root / "overlay_compare_prices.parquet", "computed in notebook from cache"),
]

precompute_status = pd.DataFrame(
    [
        {
            "Artifact": name,
            "Path": display_path(path),
            "Status": mode,
            "Exists": yes_no(path.exists()),
            "Last Modified": pd.Timestamp(path.stat().st_mtime, unit="s") if path.exists() else pd.NaT,
        }
        for name, path, mode in tracked_files
    ]
)
precompute_status

,Artifact,Path,Status,Exists,Last Modified
0,Main US sleeve metrics,result\multi_factor_copula_metrics.csv,precomputed,Yes,2026-05-29 11:36:35.688215494
1,US lookback sweep,result\static_hmm_603010_lookback_sweep.csv,precomputed,Yes,2026-05-29 11:39:15.367542267
2,US rebalance sweep,result\static_hmm_603010_rebalance_sweep.csv,precomputed,Yes,2026-05-29 11:39:25.132997990
3,US mix sweep,result\static_hmm_momentum_mix_sweep.csv,precomputed,Yes,2026-05-29 11:36:50.519509792
4,US overlay summary,result\overlay_comparison_summary.csv,precomputed,Yes,2026-05-29 11:36:46.386732340
5,Confirmed best US overlay,result\joint_confirm_603010_504d_1m_overlay_summary_usd.csv,precomputed,Yes,2026-05-29 13:12:05.057842493
6,Thailand PIT sleeve,result\thai_set100_pit_metrics.csv,precomputed,Yes,2026-05-12 13:07:42.969618559
7,US/TH stocks-only blend,result\us_th_stocks_only_blended_summary_thb.csv,precomputed,Yes,2026-05-30 03:24:12.496339798
8,US/TH stocks-only joint model,result\us_th_joint_stocks_only_pit_reselect_summary_thb.csv,precomputed,Yes,2026-05-30 02:40:59.281276226
9,US/TH Gold/BTC blend,result\us_th_gold_btc_blended_summary_thb.csv,precomputed,Yes,2026-05-30 03:24:20.561936378


## Config Registry

This is the compact registry for the experiment families used in this notebook. If a result is in this table, we treat it as part of the supported summary path rather than an orphan CSV.

In [29]:
config_registry = pd.read_csv(paths.result_dir / "test_config_registry.csv")
config_registry

,Step,Experiment,Mode,Source,Universe,Lookback Days,Rebalance,Daily Exposure,Gold/BTC Mix,Report Currency,Notes
0,0,S&P 500 buy and hold,on-the-fly from cached benchmark,source cache benchmark.parquet,SPY benchmark series,NaN,none,No,No,USD,Computed from cached benchmark price series inside the summary notebook.
1,1,US PIT equity sleeve baseline,precomputed,result/multi_factor_copula_metrics.csv,"sp500_pit, top 30 liquid names",756,ME,No,No,USD,"4 clusters, max_weight 0.08, momentum on, mom_63, feature flags resid_vol/drawdown/downside_beta off."
2,4,US overlay lookback sweep,precomputed,result/static_hmm_603010_lookback_sweep.csv,"sp500_pit, top 30 liquid names",252 / 504 / 756,ME,Yes,60/30/10,USD,"4 clusters, max_weight 0.08, momentum on, overlay strategic rebalance held at notebook baseline."
3,4,US overlay strategic rebalance sweep,precomputed,result/static_hmm_603010_rebalance_sweep.csv,"sp500_pit, top 30 liquid names",756,overlay rebalance 1 / 3 / 6 months,Yes,60/30/10,USD,Equity sleeve still monthly; sweep is the strategic overlay rebalance cadence.
4,2,Thailand PIT equity sleeve baseline,precomputed,result/thai_set100_pit_metrics.csv,"set100_pit, top 30 liquid names",504,ME,No,No,THB,"^SET.BK benchmark, no vol proxy, 4 clusters, max_weight 0.08, momentum on, mom_63."
5,2,US + TH fixed-weight stock-only blends,precomputed,result/us_th_stocks_only_blended_summary_thb.csv,US Static HMM sleeve + TH Static HMM sleeve,504,"ME sleeve, 1M strategic mix",No,No,THB,"Fixed-weight stock-only blends such as 100/0, 85/15, 70/30, 50/50, 30/70, and 0/100."
6,2,US + TH joint stocks-only model,precomputed,result/us_th_joint_stocks_only_pit_reselect_summary_thb.csv,US and TH equities in one THB model,504,ME sleeve,No,No,THB,"Joint model with US and TH equities only, full PIT reselect every rebalance, no Gold/BTC overlay."
7,3,Static HMM + Gold/BTC mix sweep,precomputed,result/static_hmm_momentum_mix_sweep.csv,US Static HMM sleeve,756,3M strategic mix,No,"100/0/0, 70/20/10, 65/25/10, 60/30/10",USD,Momentum-enabled sleeve mix sweep before the later confirmed 504d/1M update.
8,4,US/TH Gold/BTC blend family,precomputed,result/us_th_gold_btc_blended_summary_thb.csv,US and TH sleeves with Gold/BTC overlay,504,daily exposure + strategic mix,Yes,US/TH/Gold/BTC fixed allocations,THB,Blend family that already includes the daily exposure overlay on the equity and side assets.
9,4,US/TH stocks-only vs with Gold/BTC side-trigger comparison,precomputed,result/us_th_stocks_only_vs_gold_btc_side_trigger_pit_reselect_comparison_thb.csv,US + TH equity sleeves with side-trigger risk shift,504,daily exposure + fee/slippage test,Yes,compare none vs 60/30/10 overlay,THB,This file isolates how adding Gold/BTC changes the stock-only side-trigger portfolio after full PIT reselect every r...


## Step 0 - S&P Buy And Hold

This is the benchmark reference. We compute it directly from the cached benchmark price series so the notebook always has a clean baseline even if no dedicated summary CSV exists for SPY buy and hold.

In [30]:
benchmark = pd.read_parquet(paths.source_cache_root / "benchmark.parquet").rename(columns={"value": "benchmark"})
benchmark.index = pd.to_datetime(benchmark.index)
benchmark = benchmark["benchmark"].sort_index().loc["2012-01-01":"2026-04-30"].ffill()

def metrics_from_price_window(series, start, end, label):
    window = series.loc[start:end].dropna()
    returns = window.pct_change(fill_method=None).fillna(0.0)
    curve = curve_from_returns(returns)
    row = compute_port_opt_style_metrics(curve, risk_free_rate=0.03).to_dict()
    row["Strategy"] = label
    row["Start"] = window.index.min().date().isoformat()
    row["End"] = window.index.max().date().isoformat()
    return row

spx_windows = pd.DataFrame(
    [
        metrics_from_price_window(benchmark, "2012-01-01", "2026-04-30", "S&P 500 buy and hold (full)"),
        metrics_from_price_window(benchmark, "2016-01-04", "2026-04-29", "S&P 500 buy and hold (overlay window)"),
        metrics_from_price_window(benchmark, "2017-12-29", "2026-04-29", "S&P 500 buy and hold (US+TH joint window)"),
    ]
).set_index("Strategy")
spx_windows

,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Start,End
Strategy,,,,,,,,,
S&P 500 buy and hold (full),6.2364,0.1485,0.1661,0.7366,0.9056,-0.3372,0.5526,2012-01-03,2026-04-30
S&P 500 buy and hold (overlay window),3.1880,0.1492,0.1789,0.6999,0.8445,-0.3372,0.5540,2016-01-04,2026-04-29
S&P 500 buy and hold (US+TH joint window),2.0295,0.1428,0.1927,0.6340,0.7765,-0.3372,0.5531,2017-12-29,2026-04-29


## Step 1 - US PIT Static vs Dynamic HMM

This section is sleeve-only. It summarizes the main US point-in-time equity sleeve against S&P 500 buy and hold. It does not include Gold or BTC. The current precomputed family keeps `n_clusters=4`; there is no separate cluster-count sweep artifact in the repo today, so cluster count is treated as fixed rather than optimized from a saved sweep.

In [31]:
us_metrics = read_csv_clean(paths.result_dir / "multi_factor_copula_metrics.csv").set_index("Strategy")
us_vs_spx = pd.concat(
    [
        spx_windows.loc[["S&P 500 buy and hold (full)"]][["CAGR", "Annual Vol", "Sharpe", "Max Drawdown"]],
        us_metrics[["CAGR", "Annual Vol", "Sharpe", "Max Drawdown", "Turnover"]],
    ],
    axis=0,
)
display(us_vs_spx)

,CAGR,Annual Vol,Sharpe,Max Drawdown,Turnover
Strategy,,,,,
S&P 500 buy and hold (full),0.1485,0.1661,0.7366,-0.3372,NaN
Static Copula,0.2529,0.2212,1.1329,-0.3133,0.3005
Dynamic HMM Copula,0.2523,0.2214,1.1300,-0.3139,0.3001
Equal Weight,0.1784,0.2122,0.8819,-0.3513,0.0107
Risk Parity,0.1602,0.1975,0.8533,-0.3549,0.0173


## Step 2 - Add Thailand

This section answers the Thailand expansion in three layers:

- Thailand-only PIT sleeve performance
- fixed-weight US/TH stock-only blends
- US and Thailand together inside one joint stocks-only model

In [32]:
thai_metrics = read_csv_clean(paths.result_dir / "thai_set100_pit_metrics.csv").set_index("Strategy")
us_th_sleeve_compare = pd.concat(
    {
        "US PIT sleeve": us_metrics[["CAGR", "Sharpe", "Max Drawdown", "Turnover"]],
        "TH PIT sleeve": thai_metrics[["CAGR", "Sharpe", "Max Drawdown", "Turnover"]],
    },
    axis=1,
)

us_th_stock_blend = pd.read_csv(paths.result_dir / "us_th_stocks_only_blended_summary_thb.csv").sort_values("Sharpe", ascending=False)
us_th_joint_stocks_only = pd.read_csv(paths.result_dir / "us_th_joint_stocks_only_pit_reselect_summary_thb.csv").sort_values("Sharpe", ascending=False)

fixed_base = us_th_stock_blend.loc[us_th_stock_blend["Strategy"] == "US/TH stocks only 100/0"].iloc[0]
fixed_help = us_th_stock_blend.copy()
fixed_help["CAGR Delta vs 100/0"] = fixed_help["CAGR"] - float(fixed_base["CAGR"])
fixed_help["Sharpe Delta vs 100/0"] = fixed_help["Sharpe"] - float(fixed_base["Sharpe"])
fixed_help["Max DD Delta vs 100/0"] = fixed_help["Max Drawdown"] - float(fixed_base["Max Drawdown"])
fixed_help["Thailand Helped?"] = fixed_help["Sharpe Delta vs 100/0"].map(lambda x: "Yes" if x > 0 else "No")

joint_base_map = {
    "Static": us_th_joint_stocks_only.loc[us_th_joint_stocks_only["Strategy"] == "Joint US-only stocks Static Copula PIT reselect"].iloc[0],
    "Dynamic": us_th_joint_stocks_only.loc[us_th_joint_stocks_only["Strategy"] == "Joint US-only stocks Dynamic HMM Copula PIT reselect"].iloc[0],
}
joint_help_rows = []
for _, row in us_th_joint_stocks_only.iterrows():
    model_key = "Dynamic" if "Dynamic" in row["Strategy"] else "Static"
    base = joint_base_map[model_key]
    joint_help_rows.append(
        {
            "Strategy": row["Strategy"],
            "Model": model_key,
            "CAGR": row["CAGR"],
            "Sharpe": row["Sharpe"],
            "Max Drawdown": row["Max Drawdown"],
            "CAGR Delta vs US-only joint": row["CAGR"] - float(base["CAGR"]),
            "Sharpe Delta vs US-only joint": row["Sharpe"] - float(base["Sharpe"]),
            "Max DD Delta vs US-only joint": row["Max Drawdown"] - float(base["Max Drawdown"]),
            "Thailand Helped?": "Yes" if (row["Sharpe"] - float(base["Sharpe"])) > 0 else "No",
        }
    )
joint_help = pd.DataFrame(joint_help_rows)

display(us_th_sleeve_compare)
display(us_th_stock_blend)
display(us_th_joint_stocks_only)
display(fixed_help[["Strategy", "CAGR Delta vs 100/0", "Sharpe Delta vs 100/0", "Max DD Delta vs 100/0", "Thailand Helped?"]])
display(joint_help[["Strategy", "CAGR Delta vs US-only joint", "Sharpe Delta vs US-only joint", "Max DD Delta vs US-only joint", "Thailand Helped?"]])

US PIT sleeve                              TH PIT sleeve  \
                            CAGR Sharpe Max Drawdown Turnover          CAGR   
Strategy                                                                      
Static Copula             0.2529 1.1329      -0.3133   0.3005        0.0212   
Dynamic HMM Copula        0.2523 1.1300      -0.3139   0.3001        0.0222   
Equal Weight              0.1784 0.8819      -0.3513   0.0107        0.0104   
Risk Parity               0.1602 0.8533      -0.3549   0.0173        0.0146   

                                                 
                   Sharpe Max Drawdown Turnover  
Strategy                                         
Static Copula      0.2038      -0.5056   0.3306  
Dynamic HMM Copula 0.2087      -0.5033   0.3302  
Equal Weight       0.1459      -0.4905   0.0220  
Risk Parity        0.1684      -0.4555   0.0281

,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Strategy,Start,End
0,15.0746,0.2551,0.2315,0.9687,1.2867,-0.2918,0.5495,US/TH stocks only 100/0,2014-01-31,2026-04-30
1,10.5689,0.2218,0.2025,0.9436,1.2219,-0.2978,0.5534,US/TH stocks only 85/15,2014-01-31,2026-04-30
2,7.2164,0.1881,0.1780,0.8898,1.1258,-0.3038,0.5580,US/TH stocks only 70/30,2014-01-31,2026-04-30
3,4.0980,0.1426,0.1564,0.7397,0.9187,-0.3187,0.5479,US/TH stocks only 50/50,2014-01-31,2026-04-30
4,2.0861,0.0966,0.1525,0.4851,0.6000,-0.3393,0.5356,US/TH stocks only 30/70,2014-01-31,2026-04-30
5,0.3858,0.0271,0.1818,0.0733,0.0917,-0.5056,0.4820,US/TH stocks only 0/100,2014-01-31,2026-04-30


,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Strategy,Start,End,Objective,Selection Rule
0,7.1302,0.2765,0.2531,0.9740,1.3025,-0.2927,0.5430,Joint US-only stocks Dynamic HMM Copula PIT reselect,2017-12-29,2026-04-29,min_vol_mom_tilt,Full PIT reselect every rebalance
1,7.0819,0.2756,0.2528,0.9721,1.2996,-0.2930,0.5426,Joint US-only stocks Static Copula PIT reselect,2017-12-29,2026-04-29,min_vol_mom_tilt,Full PIT reselect every rebalance
2,2.5356,0.1585,0.2116,0.6604,0.8616,-0.2928,0.5379,Joint US+TH stocks only Dynamic HMM Copula PIT reselect,2017-12-29,2026-04-29,min_vol_mom_tilt,Full PIT reselect every rebalance
3,2.5327,0.1584,0.2115,0.6601,0.8613,-0.2927,0.5375,Joint US+TH stocks only Static Copula PIT reselect,2017-12-29,2026-04-29,min_vol_mom_tilt,Full PIT reselect every rebalance


,Strategy,CAGR Delta vs 100/0,Sharpe Delta vs 100/0,Max DD Delta vs 100/0,Thailand Helped?
0,US/TH stocks only 100/0,0.0000,0.0000,0.0000,No
1,US/TH stocks only 85/15,-0.0333,-0.0251,-0.0060,No
2,US/TH stocks only 70/30,-0.0671,-0.0789,-0.0120,No
3,US/TH stocks only 50/50,-0.1126,-0.2290,-0.0269,No
4,US/TH stocks only 30/70,-0.1585,-0.4836,-0.0475,No
5,US/TH stocks only 0/100,-0.2281,-0.8954,-0.2138,No


,Strategy,CAGR Delta vs US-only joint,Sharpe Delta vs US-only joint,Max DD Delta vs US-only joint,Thailand Helped?
0,Joint US-only stocks Dynamic HMM Copula PIT reselect,0.0000,0.0000,0.0000,No
1,Joint US-only stocks Static Copula PIT reselect,0.0000,0.0000,0.0000,No
2,Joint US+TH stocks only Dynamic HMM Copula PIT reselect,-0.1180,-0.3136,-0.0001,No
3,Joint US+TH stocks only Static Copula PIT reselect,-0.1172,-0.3120,0.0002,No


## Step 3 - Add Gold and BTC

This section is only for the plain allocation question: how much Gold and BTC should be added to the base US sleeve before introducing the daily exposure overlay logic.

This mix sweep uses the US Static HMM sleeve only, with no Thailand sleeve included.

It intentionally stays narrow:

- the older US Static HMM mix sweep

The all-assets-in-one-model capped portfolio is shown in its own max-weight section next, and the broader overlay families that already depend on daily exposure are shown in Step 4.

In [33]:
mix_sweep = pd.read_csv(paths.result_dir / "static_hmm_momentum_mix_sweep.csv").sort_values("Sharpe", ascending=False)

display(mix_sweep)

,Mix,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate
3,60/30/10,11.6028,0.1846,0.1381,1.0796,1.2990,-0.2405,0.4947
2,65/25/10,12.1832,0.1882,0.1441,1.0616,1.2716,-0.2523,0.4968
1,70/20/10,12.7711,0.1917,0.1506,1.0410,1.2424,-0.2641,0.4960
0,100/0/0,9.4717,0.1700,0.1862,0.7763,0.8285,-0.3133,0.3843


## Step 3B - Asset Max Weight

This section isolates the all-assets-in-one-model static portfolio for US stocks, Thailand stocks, Gold, and BTC.

The key idea here is not daily exposure. It is asset-level weight caps:

- default equity max weight = 8%
- Gold max weight = 30%
- BTC max weight = 10%

This is a one-model static portfolio with capped weights, so it belongs in its own section rather than being mixed into the plain Gold/BTC allocation sweep.

In [34]:
all_asset_static_caps = pd.read_csv(paths.result_dir / "us_th_all_asset_static_caps_pit_reselect_summary_thb.csv").rename(columns={"Unnamed: 0": "Strategy"})
all_asset_cap_sweep = pd.read_csv(paths.result_dir / "us_th_all_asset_cap_sweep_pit_reselect_summary_thb.csv").sort_values("Sharpe", ascending=False)
all_asset_static_caps_latest = pd.read_csv(paths.result_dir / "us_th_all_asset_static_caps_pit_reselect_latest_weights_thb.csv")
all_asset_cap_sweep_latest = pd.read_csv(paths.result_dir / "us_th_all_asset_cap_sweep_pit_reselect_latest_weights_thb.csv")
all_asset_static_caps.insert(1, "Daily Exposure", "No")
all_asset_cap_sweep.insert(1, "Daily Exposure", "No")
best_cap_case = "US6/TH6/Gold30/BTC10"
best_cap_last_weight = (
    all_asset_cap_sweep_latest.loc[all_asset_cap_sweep_latest["Case"] == best_cap_case]
    .loc[lambda df: df["Portfolio Weight"] > 0]
    .sort_values("Portfolio Weight", ascending=False)
    .reset_index(drop=True)
)

asset_cap_rules = pd.DataFrame(
    [
        {"Asset Group": "US Equity", "Max Weight": 0.08, "Notes": "Default per-asset cap"},
        {"Asset Group": "TH Equity", "Max Weight": 0.08, "Notes": "Default per-asset cap"},
        {"Asset Group": "Gold", "Max Weight": 0.30, "Notes": "Explicit override cap"},
        {"Asset Group": "BTC", "Max Weight": 0.10, "Notes": "Explicit override cap"},
    ]
)

display(asset_cap_rules)
display(all_asset_static_caps)
display(all_asset_cap_sweep)
display(best_cap_last_weight)
display(all_asset_static_caps_latest)

,Asset Group,Max Weight,Notes
0,US Equity,0.0800,Default per-asset cap
1,TH Equity,0.0800,Default per-asset cap
2,Gold,0.3000,Explicit override cap
3,BTC,0.1000,Explicit override cap


,Total Return,Daily Exposure,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Strategy,Start,End,Objective,Selection Rule
0,3.5326,No,0.1925,0.2070,0.8101,1.1010,-0.2730,0.5338,US/TH stocks + Gold/BTC all assets Static model capped rebalance PIT reselect,2017-12-29,2026-04-29,mean_variance,Full PIT reselect every rebalance


,Total Return,Daily Exposure,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Strategy,Start,End,Objective,US Equity Max Weight,TH Equity Max Weight,Gold Max Weight,BTC Max Weight,Selection Rule
0,3.8921,No,0.2032,0.1906,0.9097,1.2036,-0.2751,0.5379,All-assets static capped [US6/TH6/Gold30/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.0600,0.0600,0.3000,0.1000,Full PIT reselect every rebalance
1,3.8320,No,0.2014,0.1900,0.9042,1.2199,-0.2602,0.5375,All-assets static capped [US6/TH6/Gold40/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.0600,0.0600,0.4000,0.1000,Full PIT reselect every rebalance
2,3.7577,No,0.1993,0.1929,0.8842,1.1579,-0.2853,0.5389,All-assets static capped [US6/TH6/Gold20/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.0600,0.0600,0.2000,0.1000,Full PIT reselect every rebalance
3,3.2061,No,0.1822,0.1859,0.8329,1.1031,-0.2645,0.5393,All-assets static capped [US6/TH6/Gold30/BTC5] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.0600,0.0600,0.3000,0.0500,Full PIT reselect every rebalance
4,3.1535,No,0.1805,0.1846,0.8294,1.1153,-0.2518,0.5393,All-assets static capped [US6/TH6/Gold40/BTC5] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.0600,0.0600,0.4000,0.0500,Full PIT reselect every rebalance
5,3.1943,No,0.1818,0.1889,0.8209,1.0768,-0.2726,0.5402,All-assets static capped [US6/TH6/Gold20/BTC5] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.0600,0.0600,0.2000,0.0500,Full PIT reselect every rebalance
6,3.7721,No,0.1997,0.2206,0.8006,1.0839,-0.2769,0.5338,All-assets static capped [US10/TH6/Gold40/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.1000,0.0600,0.4000,0.1000,Full PIT reselect every rebalance
7,3.7817,No,0.2000,0.2221,0.7979,1.0796,-0.2767,0.5375,All-assets static capped [US10/TH6/Gold30/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.1000,0.0600,0.3000,0.1000,Full PIT reselect every rebalance
8,3.8046,No,0.2006,0.2246,0.7940,1.0677,-0.2823,0.5365,All-assets static capped [US10/TH6/Gold20/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.1000,0.0600,0.2000,0.1000,Full PIT reselect every rebalance
9,3.4495,No,0.1900,0.2164,0.7741,1.0560,-0.2604,0.5319,All-assets static capped [US10/TH6/Gold40/BTC5] PIT reselect,2017-12-29,2026-04-29,mean_variance,0.1000,0.0600,0.4000,0.0500,Full PIT reselect every rebalance


,Asset,Portfolio Weight,Date,Case,US Equity Max Weight,TH Equity Max Weight,Gold Max Weight,BTC Max Weight,Sleeve
0,PTT.BK,0.0600,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,TH Equity
1,CPN.BK,0.0600,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,TH Equity
2,WMT,0.0600,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,US Equity
3,IVL.BK,0.0600,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,TH Equity
4,COST,0.0600,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,US Equity
...,...,...,...,...,...,...,...,...,...
57,TISCO.BK,0.0000,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,TH Equity
58,SCB.BK,0.0000,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,TH Equity
59,EA.BK,0.0000,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,TH Equity
60,SCC.BK,0.0000,2026-03-31,US6/TH6/Gold30/BTC10,0.0600,0.0600,0.3000,0.1000,TH Equity


,Asset,Portfolio Weight,Date,Sleeve
0,AAPL,0.0000,2026-03-31,US Equity
1,ADBE,0.0000,2026-03-31,US Equity
2,ADVANC.BK,0.0800,2026-03-31,TH Equity
3,AMATA.BK,0.0000,2026-03-31,TH Equity
4,AMD,0.0000,2026-03-31,US Equity
...,...,...,...,...
106,V,0.0000,2026-03-31,US Equity
107,WFC,0.0000,2026-03-31,US Equity
108,WHA.BK,0.0000,2026-03-31,TH Equity
109,WMT,0.0000,2026-03-31,US Equity


## Step 3C - Top 5 Cap Cases by Objective

This section takes the top 5 all-asset cap cases by Sharpe and reruns each case across the objective menu:

- mean_variance
- max_sharpe_mom
- min_vol_mom_tilt
- risk_parity_mom_tilt

The goal is to separate two questions:

- which cap structure is best
- which objective is best once that cap structure is fixed

In [35]:
all_asset_cap_top5_objective = pd.read_csv(paths.result_dir / "us_th_all_asset_cap_top5_objective_pit_reselect_sweep_thb.csv")
all_asset_cap_top5_best = pd.read_csv(paths.result_dir / "us_th_all_asset_cap_top5_objective_pit_reselect_best_by_case_thb.csv")
all_asset_cap_top5_latest = pd.read_csv(paths.result_dir / "us_th_all_asset_cap_top5_objective_pit_reselect_latest_weights_thb.csv")
best_top5_strategy = all_asset_cap_top5_best.sort_values("Sharpe", ascending=False).iloc[0]["Strategy"]
best_top5_last_weight = (
    all_asset_cap_top5_latest.loc[all_asset_cap_top5_latest["Strategy"] == best_top5_strategy]
    .loc[lambda df: df["Portfolio Weight"] > 0]
    .sort_values("Portfolio Weight", ascending=False)
    .reset_index(drop=True)
)

display(all_asset_cap_top5_best)
display(all_asset_cap_top5_objective.sort_values(["Top 5 Rank", "Sharpe"], ascending=[True, False]))
display(best_top5_last_weight)

,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Strategy,Base Case,Top 5 Rank,Objective,Start,End,US Equity Max Weight,TH Equity Max Weight,Gold Max Weight,BTC Max Weight,Selection Rule,Source Case Sharpe,Sharpe Delta vs Base Leader
0,3.8921,0.2032,0.1906,0.9097,1.2036,-0.2751,0.5379,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,2017-12-29,2026-04-29,0.0600,0.0600,0.3000,0.1000,Full PIT reselect every rebalance,0.9097,0.0000
1,3.8320,0.2014,0.1900,0.9042,1.2199,-0.2602,0.5375,All-assets static capped [US6/TH6/Gold40/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold40/BTC10,2,mean_variance,2017-12-29,2026-04-29,0.0600,0.0600,0.4000,0.1000,Full PIT reselect every rebalance,0.9042,0.0000
2,3.7577,0.1993,0.1929,0.8842,1.1579,-0.2853,0.5389,All-assets static capped [US6/TH6/Gold20/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold20/BTC10,3,mean_variance,2017-12-29,2026-04-29,0.0600,0.0600,0.2000,0.1000,Full PIT reselect every rebalance,0.8842,0.0000
3,3.2061,0.1822,0.1859,0.8329,1.1031,-0.2645,0.5393,All-assets static capped [US6/TH6/Gold30/BTC5] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC5,4,mean_variance,2017-12-29,2026-04-29,0.0600,0.0600,0.3000,0.0500,Full PIT reselect every rebalance,0.8329,0.0000
4,3.2046,0.1821,0.1851,0.8354,1.1319,-0.2495,0.5365,All-assets static capped [US6/TH6/Gold40/BTC5] [min_vol_mom_tilt] PIT reselect,US6/TH6/Gold40/BTC5,5,min_vol_mom_tilt,2017-12-29,2026-04-29,0.0600,0.0600,0.4000,0.0500,Full PIT reselect every rebalance,0.8294,0.0060


,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Strategy,Base Case,Top 5 Rank,Objective,Start,End,US Equity Max Weight,TH Equity Max Weight,Gold Max Weight,BTC Max Weight,Selection Rule
0,3.8921,0.2032,0.1906,0.9097,1.2036,-0.2751,0.5379,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,2017-12-29,2026-04-29,0.0600,0.0600,0.3000,0.1000,Full PIT reselect every rebalance
1,3.7841,0.2001,0.1907,0.8957,1.1908,-0.2731,0.5370,All-assets static capped [US6/TH6/Gold30/BTC10] [min_vol_mom_tilt] PIT reselect,US6/TH6/Gold30/BTC10,1,min_vol_mom_tilt,2017-12-29,2026-04-29,0.0600,0.0600,0.3000,0.1000,Full PIT reselect every rebalance
2,1.0000,0.0841,0.1518,0.4111,0.5164,-0.2935,0.5278,All-assets static capped [US6/TH6/Gold30/BTC10] [max_sharpe_mom] PIT reselect,US6/TH6/Gold30/BTC10,1,max_sharpe_mom,2017-12-29,2026-04-29,0.0600,0.0600,0.3000,0.1000,Full PIT reselect every rebalance
3,0.7990,0.0708,0.1477,0.3350,0.3929,-0.3338,0.5310,All-assets static capped [US6/TH6/Gold30/BTC10] [risk_parity_mom_tilt] PIT reselect,US6/TH6/Gold30/BTC10,1,risk_parity_mom_tilt,2017-12-29,2026-04-29,0.0600,0.0600,0.3000,0.1000,Full PIT reselect every rebalance
4,3.8320,0.2014,0.1900,0.9042,1.2199,-0.2602,0.5375,All-assets static capped [US6/TH6/Gold40/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold40/BTC10,2,mean_variance,2017-12-29,2026-04-29,0.0600,0.0600,0.4000,0.1000,Full PIT reselect every rebalance
5,3.7335,0.1986,0.1894,0.8938,1.2075,-0.2608,0.5365,All-assets static capped [US6/TH6/Gold40/BTC10] [min_vol_mom_tilt] PIT reselect,US6/TH6/Gold40/BTC10,2,min_vol_mom_tilt,2017-12-29,2026-04-29,0.0600,0.0600,0.4000,0.1000,Full PIT reselect every rebalance
6,1.0594,0.0878,0.1493,0.4383,0.5601,-0.2823,0.5278,All-assets static capped [US6/TH6/Gold40/BTC10] [max_sharpe_mom] PIT reselect,US6/TH6/Gold40/BTC10,2,max_sharpe_mom,2017-12-29,2026-04-29,0.0600,0.0600,0.4000,0.1000,Full PIT reselect every rebalance
7,0.7990,0.0708,0.1477,0.3350,0.3929,-0.3338,0.5310,All-assets static capped [US6/TH6/Gold40/BTC10] [risk_parity_mom_tilt] PIT reselect,US6/TH6/Gold40/BTC10,2,risk_parity_mom_tilt,2017-12-29,2026-04-29,0.0600,0.0600,0.4000,0.1000,Full PIT reselect every rebalance
8,3.7577,0.1993,0.1929,0.8842,1.1579,-0.2853,0.5389,All-assets static capped [US6/TH6/Gold20/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold20/BTC10,3,mean_variance,2017-12-29,2026-04-29,0.0600,0.0600,0.2000,0.1000,Full PIT reselect every rebalance
9,3.6667,0.1966,0.1925,0.8741,1.1449,-0.2846,0.5398,All-assets static capped [US6/TH6/Gold20/BTC10] [min_vol_mom_tilt] PIT reselect,US6/TH6/Gold20/BTC10,3,min_vol_mom_tilt,2017-12-29,2026-04-29,0.0600,0.0600,0.2000,0.1000,Full PIT reselect every rebalance


,Asset,Portfolio Weight,Date,Strategy,Base Case,Top 5 Rank,Objective,US Equity Max Weight,TH Equity Max Weight,Gold Max Weight,BTC Max Weight,Sleeve
0,PTT.BK,0.0600,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,TH Equity
1,CPN.BK,0.0600,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,TH Equity
2,WMT,0.0600,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,US Equity
3,IVL.BK,0.0600,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,TH Equity
4,COST,0.0600,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,US Equity
...,...,...,...,...,...,...,...,...,...,...,...,...
57,TISCO.BK,0.0000,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,TH Equity
58,SCB.BK,0.0000,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,TH Equity
59,EA.BK,0.0000,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,TH Equity
60,SCC.BK,0.0000,2026-03-31,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,US6/TH6/Gold30/BTC10,1,mean_variance,0.0600,0.0600,0.3000,0.1000,TH Equity


## Step 3D - Best Cap Config with More Stock Selections

This section keeps the best cap-weight setup fixed and only changes how many US and Thailand stocks are allowed into the all-asset model.

Fixed best-cap configuration:

- objective = `mean_variance`
- US equity cap = 6%
- TH equity cap = 6%
- Gold cap = 40%
- BTC cap = 10%

Stock selection rule:

- this repo does **not** select by market cap
- it reselects the US and Thailand stock universes at **every rebalance date**
- inside each rebalance window it ranks candidates by median `price * volume`
- it requires strong history coverage before a name is eligible

This is the strict level-3 version of the test:

- point-in-time membership is refreshed every rebalance
- liquidity ranking is refreshed every rebalance
- then the all-asset static capped model is rebuilt on that current PIT universe

The sweep below tests whether expanding the reselected US/TH stock lists from `30/30` to `40/40`, `50/50`, and `100/100` improves the best cap-weight portfolio.

In [36]:
best_cap_asset_count_sweep = pd.read_csv(paths.result_dir / "us_th_best_cap_asset_count_pit_reselect_sweep_summary_thb.csv").sort_values("Sharpe", ascending=False)
best_cap_asset_count_latest = pd.read_csv(paths.result_dir / "us_th_best_cap_asset_count_pit_reselect_sweep_latest_weights_thb.csv")
best_cap_asset_count_winner = best_cap_asset_count_sweep.iloc[0]["Strategy"]
best_cap_asset_count_last_weight = (
    best_cap_asset_count_latest.loc[best_cap_asset_count_latest["Strategy"] == best_cap_asset_count_winner]
    .loc[lambda df: df["Portfolio Weight"] > 0]
    .sort_values("Portfolio Weight", ascending=False)
    .reset_index(drop=True)
)

stock_selection_rule = pd.DataFrame(
    [
        {"Rule": "Primary rank metric", "Value": "Median(price * volume) inside each rebalance lookback window"},
        {"Rule": "Availability filter", "Value": "At least 90% non-null price history inside the rebalance lookback window"},
        {"Rule": "Refresh cadence", "Value": "Full PIT reselect every rebalance"},
        {"Rule": "Uses market cap?", "Value": "No"},
    ]
)

display(stock_selection_rule)
display(best_cap_asset_count_sweep)
display(best_cap_asset_count_last_weight)

,Rule,Value
0,Primary rank metric,Median(price * volume) inside each rebalance lookback window
1,Availability filter,At least 90% non-null price history inside the rebalance lookback window
2,Refresh cadence,Full PIT reselect every rebalance
3,Uses market cap?,No


,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Strategy,Start,End,Objective,US Assets,TH Assets,US Equity Max Weight,TH Equity Max Weight,Gold Max Weight,BTC Max Weight,Selection Rule
0,3.6765,0.1969,0.1903,0.8831,1.1897,-0.2604,0.5379,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,30,30,0.0600,0.0600,0.4000,0.1000,"PIT reselect every rebalance using median dollar volume, availability >= 90%"
1,3.4454,0.1898,0.1899,0.8533,1.1792,-0.2555,0.5356,All-assets static capped [US50/TH50/Gold40/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,50,50,0.0600,0.0600,0.4000,0.1000,"PIT reselect every rebalance using median dollar volume, availability >= 90%"
2,3.3937,0.1882,0.1905,0.8442,1.1542,-0.2563,0.5421,All-assets static capped [US40/TH40/Gold40/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,40,40,0.0600,0.0600,0.4000,0.1000,"PIT reselect every rebalance using median dollar volume, availability >= 90%"
3,0.6153,0.0575,0.1714,0.2372,0.3166,-0.3392,0.5379,All-assets static capped [US100/TH100/Gold40/BTC10] PIT reselect,2017-12-29,2026-04-29,mean_variance,100,100,0.0600,0.0600,0.4000,0.1000,"PIT reselect every rebalance using median dollar volume, availability >= 90%"


,Asset,Portfolio Weight,Date,US Assets,TH Assets,Strategy,Objective,US Equity Max Weight,TH Equity Max Weight,Gold Max Weight,BTC Max Weight,Sleeve
0,ADVANC.BK,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,TH Equity
1,PTTGC.BK,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,TH Equity
2,PTTEP.BK,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,TH Equity
3,WMT,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,US Equity
4,CPN.BK,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,TH Equity
5,KTB.BK,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,TH Equity
6,IVL.BK,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,TH Equity
7,MU,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,US Equity
8,INTC,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,US Equity
9,HANA.BK,0.0600,2026-03-31,30,30,All-assets static capped [US30/TH30/Gold40/BTC10] PIT reselect,mean_variance,0.0600,0.0600,0.4000,0.1000,TH Equity


## Step 4 - Daily Exposure

Daily exposure in this repo must be non-lookahead:

- the signal is observed from today's close and today's fully known state
- that signal affects tomorrow's exposure
- we do not let today's close change today's return

The helper functions in `src/dynamic_factor_copula.py` now lag close-based signals by one session before applying them to returns.

This section contains all summary tables that depend on daily exposure overlays, including:

- the confirmed best US overlay summary
- US/TH Gold/BTC blend rows
- direct stocks-only vs with-Gold/BTC side-trigger comparison

The primary US overlay table shown here is the confirmed best config:

- equity sleeve lookback = 504 trading days
- overlay strategic rebalance = 1 month

The older legacy `756d / 3M` summary is intentionally not shown here to keep the section focused on the current preferred baseline.

In [37]:
confirmed_us = read_csv_clean(paths.result_dir / "joint_confirm_603010_504d_1m_overlay_summary_usd.csv").set_index("Strategy")
us_th_gold_btc_blend = pd.read_csv(paths.result_dir / "us_th_gold_btc_blended_summary_thb.csv").sort_values("Sharpe", ascending=False)
stocks_only_vs_gold_btc = pd.read_csv(paths.result_dir / "us_th_stocks_only_vs_gold_btc_side_trigger_pit_reselect_comparison_thb.csv").sort_values("Sharpe", ascending=False)
best_config = json.load(open(paths.result_dir / "best_config_latest.json", encoding="utf-8"))

overlay_prices = load_overlay_compare_prices(
    paths,
    start_date="2016-01-01",
    end_date="2026-04-29",
    tickers=["SPY", "GC=F", "BTC-USD", "^VIX", "USDTHB=X"],
).dropna(subset=["SPY", "GC=F", "BTC-USD", "^VIX"])

spy = overlay_prices["SPY"]
gold = overlay_prices["GC=F"]
btc = overlay_prices["BTC-USD"]
vix = overlay_prices["^VIX"]
zero_fx = pd.Series(0.0, index=overlay_prices.index, dtype=float)

def daily_exposure_delta(label, raw_returns, daily_returns):
    raw_curve = curve_from_returns(raw_returns)
    daily_curve = curve_from_returns(daily_returns)
    raw_metrics = compute_port_opt_style_metrics(raw_curve, risk_free_rate=0.03)
    daily_metrics = compute_port_opt_style_metrics(daily_curve, risk_free_rate=0.03)
    delta = (daily_metrics - raw_metrics).to_dict()
    delta["Asset"] = label
    delta["Sharpe Helped"] = yes_no(delta["Sharpe"] > 0)
    delta["Drawdown Helped"] = yes_no(delta["Max Drawdown"] > 0)
    return delta

spy_returns = spy.pct_change(fill_method=None).fillna(0.0)
gold_returns = gold.pct_change(fill_method=None).fillna(0.0)
btc_returns = btc.pct_change(fill_method=None).fillna(0.0)

spy_daily = compare_apply_returns(spy_returns, compare_sp_exposure(spy, vix), "USD_STATIC", zero_fx)
gold_daily = compare_apply_returns(gold_returns, compare_trend_exposure(gold, 0.50), "USD_STATIC", zero_fx)
btc_daily = compare_apply_returns(btc_returns, compare_trend_exposure(btc, 0.00), "USD_STATIC", zero_fx)

asset_daily_exposure_test = pd.DataFrame(
    [
        daily_exposure_delta("SPY", spy_returns, spy_daily),
        daily_exposure_delta("Gold", gold_returns, gold_daily),
        daily_exposure_delta("BTC", btc_returns, btc_daily),
    ]
).set_index("Asset")

confirmed_us_display = confirmed_us.copy()
confirmed_us_display.insert(
    0,
    "Momentum",
    [
        "No",
        "No",
        "Yes",
        "Yes",
    ],
)

us_th_gold_btc_blend_display = us_th_gold_btc_blend.copy()
us_th_gold_btc_blend_display.insert(1, "Momentum", "Yes")

stocks_only_vs_gold_btc_display = stocks_only_vs_gold_btc.copy()
stocks_only_vs_gold_btc_display.insert(1, "Momentum", "Yes")

asset_daily_exposure_test_display = asset_daily_exposure_test.copy()
asset_daily_exposure_test_display.insert(0, "Momentum", "No")

display(confirmed_us_display)
display(us_th_gold_btc_blend_display)
display(stocks_only_vs_gold_btc_display)
display(asset_daily_exposure_test_display)
display(pd.DataFrame([best_config["best_confirmed_joint_config"]["equity_sleeve"]]))

,Momentum,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate
Strategy,,,,,,,,
S&P 500 daily exposure,No,1.4766,0.0625,0.0999,0.3570,0.3561,-0.1922,0.3814
S&P/Gold/BTC 60/30/10 daily exposure,No,3.9895,0.1135,0.0902,0.9045,1.0805,-0.1839,0.4934
Static HMM daily exposure,Yes,4.8209,0.1250,0.1457,0.6758,0.7214,-0.2351,0.3864
Static HMM/Gold/BTC 60/30/10 daily exposure,Yes,7.4151,0.1531,0.1118,1.0622,1.2856,-0.1921,0.4971


,Strategy,Momentum,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Start,End
0,US HMM/Gold/BTC 60/30/10,Yes,6.1795,0.2110,0.1500,1.1522,1.6606,-0.1626,0.5486,2016-01-04,2026-04-29
1,US/TH/Gold/BTC 50/10/30/10,Yes,5.0505,0.1910,0.1344,1.1458,1.6506,-0.1504,0.5528,2016-01-04,2026-04-29
2,US/TH/Gold/BTC 45/15/30/10,Yes,4.5476,0.1810,0.1271,1.1379,1.6402,-0.1442,0.5520,2016-01-04,2026-04-29
3,US/TH/Gold/BTC 40/20/30/10,Yes,4.0823,0.1710,0.1201,1.1256,1.6241,-0.1381,0.5547,2016-01-04,2026-04-29
4,US/TH/Gold/BTC 30/30/30/10,Yes,3.2549,0.1510,0.1077,1.0815,1.5606,-0.1260,0.5517,2016-01-04,2026-04-29


,Strategy,Momentum,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Start,End,Objective,Fee Bps,Selection Rule,Reallocate Stock Sleeve,Avg Monthly Traded Notional,CAGR Delta vs Stocks Only,Sharpe Delta vs Stocks Only,Max DD Delta vs Stocks Only
0,"Side trigger realloc to active stock side, fee+slippage PIT reselect",Yes,2.1875,0.1446,0.1414,0.8144,1.1284,-0.2115,0.5426,2017-12-29,2026-04-29,min_vol_mom_tilt,17.0000,Full PIT reselect every rebalance,True,0.3600,0.0176,0.2338,0.0471
1,US/TH stocks only reduce risk shift active market fee+slippage PIT reselect,Yes,1.7901,0.1270,0.1834,0.5806,0.7792,-0.2586,0.5352,2017-12-29,2026-04-29,min_vol_mom_tilt,17.0000,Full PIT reselect every rebalance,True,0.6308,0.0000,0.0000,0.0000


,Momentum,Total Return,CAGR,Annual Vol,Sharpe,Sortino,Max Drawdown,Hit Rate,Sharpe Helped,Drawdown Helped
Asset,,,,,,,,,,
SPY,No,-1.7115,-0.0380,-0.0486,-0.1605,-0.1623,0.1450,0.0000,No,Yes
Gold,No,-0.6497,-0.0122,-0.0088,-0.0580,-0.0889,-0.0206,0.0000,No,No
BTC,No,-39.9658,-0.0243,-0.1131,0.0469,-0.1679,0.1349,-0.1879,Yes,Yes


,universe_mode,n_assets,selection_rule,n_clusters,lookback_days,rebalance_freq,max_weight,include_momentum_features,include_momentum_signal,momentum_signal_mode,feature_flags
0,sp500_pit,30,top liquid S&P 500 members as of each rebalance date,4,504,ME,0.0800,True,True,mom_63,"{'resid_vol': False, 'drawdown': False, 'downside_beta': False}"


## Best Sharpe

This is the compact scoreboard for the highest-Sharpe result in each headline category after the latest reruns.

In [38]:
best_sharpe_snapshot = pd.DataFrame(
    [
        {
            "Category": "Buy and hold",
            "Strategy": "S&P 500 buy and hold (full)",
            "Sharpe": float(spx_windows.loc["S&P 500 buy and hold (full)", "Sharpe"]),
            "CAGR": float(spx_windows.loc["S&P 500 buy and hold (full)", "CAGR"]),
            "Source": "benchmark cache",
        },
        {
            "Category": "US stock",
            "Strategy": us_metrics["Sharpe"].idxmax(),
            "Sharpe": float(us_metrics["Sharpe"].max()),
            "CAGR": float(us_metrics.loc[us_metrics["Sharpe"].idxmax(), "CAGR"]),
            "Source": "result/multi_factor_copula_metrics.csv",
        },
        {
            "Category": "Momentum",
            "Strategy": f"Static HMM mix {mix_sweep.iloc[0]['Mix']}",
            "Sharpe": float(mix_sweep.iloc[0]["Sharpe"]),
            "CAGR": float(mix_sweep.iloc[0]["CAGR"]),
            "Source": "result/static_hmm_momentum_mix_sweep.csv",
        },
        {
            "Category": "Daily exposure",
            "Strategy": confirmed_us["Sharpe"].idxmax(),
            "Sharpe": float(confirmed_us["Sharpe"].max()),
            "CAGR": float(confirmed_us.loc[confirmed_us["Sharpe"].idxmax(), "CAGR"]),
            "Source": "result/joint_confirm_603010_504d_1m_overlay_summary_usd.csv",
        },
        {
            "Category": "All",
            "Strategy": all_asset_cap_sweep.iloc[0]["Strategy"],
            "Sharpe": float(all_asset_cap_sweep.iloc[0]["Sharpe"]),
            "CAGR": float(all_asset_cap_sweep.iloc[0]["CAGR"]),
            "Source": "result/us_th_all_asset_cap_sweep_pit_reselect_summary_thb.csv",
        },
        {
            "Category": "Top-5 cap objective winner",
            "Strategy": all_asset_cap_top5_best.sort_values("Sharpe", ascending=False).iloc[0]["Strategy"],
            "Sharpe": float(all_asset_cap_top5_best["Sharpe"].max()),
            "CAGR": float(all_asset_cap_top5_best.sort_values("Sharpe", ascending=False).iloc[0]["CAGR"]),
            "Source": "result/us_th_all_asset_cap_top5_objective_pit_reselect_best_by_case_thb.csv",
        },
    ]
)
best_sharpe_snapshot

,Category,Strategy,Sharpe,CAGR,Source
0,Buy and hold,S&P 500 buy and hold (full),0.7366,0.1485,benchmark cache
1,US stock,Static Copula,1.1329,0.2529,result/multi_factor_copula_metrics.csv
2,Momentum,Static HMM mix 60/30/10,1.0796,0.1846,result/static_hmm_momentum_mix_sweep.csv
3,Daily exposure,Static HMM/Gold/BTC 60/30/10 daily exposure,1.0622,0.1531,result/joint_confirm_603010_504d_1m_overlay_summary_usd.csv
4,All,All-assets static capped [US6/TH6/Gold30/BTC10] PIT reselect,0.9097,0.2032,result/us_th_all_asset_cap_sweep_pit_reselect_summary_thb.csv
5,Top-5 cap objective winner,All-assets static capped [US6/TH6/Gold30/BTC10] [mean_variance] PIT reselect,0.9097,0.2032,result/us_th_all_asset_cap_top5_objective_pit_reselect_best_by_case_thb.csv


## Step 5 - Summary

This final section condenses the best currently saved answers from the repo. It is not meant to replace the detailed sections above; it is the short scoreboard for deciding what to trust first.

In [39]:
summary_scoreboard = pd.DataFrame(
    [
        {
            "Question": "Best saved US PIT sleeve family",
            "Answer": "Static Copula is slightly ahead of Dynamic HMM in the saved US PIT baseline file.",
            "Evidence": "result/multi_factor_copula_metrics.csv",
        },
        {
            "Question": "Current preferred confirmed US overlay config",
            "Answer": "Static HMM with momentum + Gold/BTC 60/30/10, 504d lookback, 1M strategic rebalance",
            "Evidence": "result/best_config_latest.json",
        },
        {
            "Question": "Best saved plain Gold/BTC mix in the older US sweep",
            "Answer": mix_sweep.iloc[0]["Mix"],
            "Evidence": "result/static_hmm_momentum_mix_sweep.csv",
        },
        {
            "Question": "Best saved US/TH fixed-weight blend in THB by Sharpe",
            "Answer": us_th_stock_blend.iloc[0]["Strategy"],
            "Evidence": "result/us_th_stocks_only_blended_summary_thb.csv",
        },
        {
            "Question": "Best saved US/TH joint stocks-only model row in THB by Sharpe",
            "Answer": us_th_joint_stocks_only.iloc[0]["Strategy"],
            "Evidence": "result/us_th_joint_stocks_only_pit_reselect_summary_thb.csv",
        },
        {
            "Question": "Does Thailand help in the stock-only fixed-weight blends?",
            "Answer": "No in the saved tests; 100/0 remains the best Sharpe and CAGR among the stock-only blend rows.",
            "Evidence": "result/us_th_stocks_only_blended_summary_thb.csv",
        },
        {
            "Question": "Does Thailand help in the stock-only joint model?",
            "Answer": "No in the saved control test; the US-only joint rows beat the US+TH joint rows on both CAGR and Sharpe.",
            "Evidence": "result/us_th_joint_stocks_only_pit_reselect_summary_thb.csv",
        },
        {
            "Question": "What happens when Gold/BTC is added to the stock-only side-trigger portfolio?",
            "Answer": stocks_only_vs_gold_btc.iloc[0]["Strategy"],
            "Evidence": "result/us_th_stocks_only_vs_gold_btc_side_trigger_pit_reselect_comparison_thb.csv",
        },
        {
            "Question": "Best saved US/TH Gold/BTC overlay blend in THB by Sharpe",
            "Answer": us_th_gold_btc_blend.iloc[0]["Strategy"],
            "Evidence": "result/us_th_gold_btc_blended_summary_thb.csv",
        },
        {
            "Question": "Cluster-count sweep available?",
            "Answer": "No saved cluster-count sweep found; current precomputed families use 4 clusters.",
            "Evidence": "config registry + existing result files",
        },
        {
            "Question": "Daily exposure asset tests fully precomputed?",
            "Answer": "No; SPY/Gold/BTC asset-level tests are computed here from cached prices, while sleeve-level overlay rows are precomputed.",
            "Evidence": "overlay cache + overlay summary files",
        },
    ]
)
summary_scoreboard

,Question,Answer,Evidence
0,Best saved US PIT sleeve family,Static Copula is slightly ahead of Dynamic HMM in the saved US PIT baseline file.,result/multi_factor_copula_metrics.csv
1,Current preferred confirmed US overlay config,"Static HMM with momentum + Gold/BTC 60/30/10, 504d lookback, 1M strategic rebalance",result/best_config_latest.json
2,Best saved plain Gold/BTC mix in the older US sweep,60/30/10,result/static_hmm_momentum_mix_sweep.csv
3,Best saved US/TH fixed-weight blend in THB by Sharpe,US/TH stocks only 100/0,result/us_th_stocks_only_blended_summary_thb.csv
4,Best saved US/TH joint stocks-only model row in THB by Sharpe,Joint US-only stocks Dynamic HMM Copula PIT reselect,result/us_th_joint_stocks_only_pit_reselect_summary_thb.csv
5,Does Thailand help in the stock-only fixed-weight blends?,No in the saved tests; 100/0 remains the best Sharpe and CAGR among the stock-only blend rows.,result/us_th_stocks_only_blended_summary_thb.csv
6,Does Thailand help in the stock-only joint model?,No in the saved control test; the US-only joint rows beat the US+TH joint rows on both CAGR and Sharpe.,result/us_th_joint_stocks_only_pit_reselect_summary_thb.csv
7,What happens when Gold/BTC is added to the stock-only side-trigger portfolio?,"Side trigger realloc to active stock side, fee+slippage PIT reselect",result/us_th_stocks_only_vs_gold_btc_side_trigger_pit_reselect_comparison_thb.csv
8,Best saved US/TH Gold/BTC overlay blend in THB by Sharpe,US HMM/Gold/BTC 60/30/10,result/us_th_gold_btc_blended_summary_thb.csv
9,Cluster-count sweep available?,No saved cluster-count sweep found; current precomputed families use 4 clusters.,config registry + existing result files
